In [1]:
##### Exposure distribution analysis: % of capital, labor, production at each deprivation percentile
# for overall agriculture and for exposed areas only

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import xarray as xr
import rasterio.features
from rasterio.enums import Resampling

In [2]:
##### SET-UP

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from helpers.reproject_raster import reproject_raster, array_to_raster

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results" / "30x30"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

##### LOAD DATA

# capital / labor / production rasters
capital_path = ROOT / "Results/Raster_models/rescaled_capital_USD.tif"
capital_path_p10 = ROOT / "Results/Raster_models/rescaled_capital_USD_p10.tif"
capital_path_p90 = ROOT / "Results/Raster_models/rescaled_capital_USD_p90.tif"

labor_path = ROOT / "Results/Raster_models/rescaled_jobs.tif"
labor_path_p10 = ROOT / "Results/Raster_models/rescaled_jobs_p10.tif"
labor_path_p90 = ROOT / "Results/Raster_models/rescaled_jobs_p90.tif"

production_path = DATA_DIR / "total_production_tonnes_2020.tif"

# relative deprivation percentile rasters (from the percentile script)
rd_percentiles_national = RESULTS_DIR / "relative_deprivation_percentiles_country.tif"
rd_percentiles_global = RESULTS_DIR / "relative_deprivation_percentiles_global.tif"

# priority raster (0 = not in top 30%, 1 = in top 30% not PA, 2 = in top 30% + PA)
biodiversity_with_PA = DATA_DIR / "conservation_scenario.tif"

# country boundaries (GADM 4.1: https://gadm.org/data.html)
countries = gpd.read_file(DATA_DIR / "gadm_410-levels.gpkg", layer="ADM_0")

In [3]:
##### ALIGN RASTERS

def load_da(x):
    return x if isinstance(x, xr.DataArray) else rioxarray.open_rasterio(x, masked=True)

def match_grid(da, target):
    return da.rio.reproject_match(target, resampling=Resampling.nearest)

# percentile rasters define the target grid
rd_national_da = rioxarray.open_rasterio(rd_percentiles_national, masked=True)
rd_global_da = match_grid(rioxarray.open_rasterio(rd_percentiles_global, masked=True), rd_national_da)
biodiversity_da = match_grid(rioxarray.open_rasterio(biodiversity_with_PA, masked=True), rd_national_da)

# resample to the conservation scenario grid (sum), then match to the percentile grid
def load_summed(path):
    da = load_da(array_to_raster(reproject_raster(str(path), resampling="sum")))
    return match_grid(da, rd_national_da)

capital_da = load_summed(capital_path)
capital_p10_da = load_summed(capital_path_p10)
capital_p90_da = load_summed(capital_path_p90)
labor_da = load_summed(labor_path)
labor_p10_da = load_summed(labor_path_p10)
labor_p90_da = load_summed(labor_path_p90)
production_da = load_summed(production_path)

def to_array(da):
    arr = da.squeeze().values.astype(float)
    nodata = da.rio.nodata
    if nodata is not None and not np.isnan(nodata):
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

pctl_arrays = {
    "global":   to_array(rd_global_da),
    "national": to_array(rd_national_da),
}

# variables and their estimates (production has no uncertainty estimates)
VARIABLES = {
    "Capital":    {"core": to_array(capital_da), "p10": to_array(capital_p10_da), "p90": to_array(capital_p90_da)},
    "Labor":      {"core": to_array(labor_da),   "p10": to_array(labor_p10_da),   "p90": to_array(labor_p90_da)},
    "Production": {"core": to_array(production_da)},
}

In [4]:
##### MASKS

# country ID raster on the percentile grid: centroid inside country first, then all_touched for coastal pixels
ref_transform = rd_national_da.rio.transform()
ref_shape = rd_national_da.squeeze().shape
countries_proj = countries.to_crs(rd_national_da.rio.crs).reset_index(drop=True)
shapes = [(geom, i) for i, geom in enumerate(countries_proj.geometry) if geom is not None]

country_id_arr = rasterio.features.rasterize(
    shapes, out_shape=ref_shape, transform=ref_transform, fill=-1, dtype="int32", all_touched=False
)
country_id_touched = rasterio.features.rasterize(
    shapes, out_shape=ref_shape, transform=ref_transform, fill=-1, dtype="int32", all_touched=True
)
country_id_arr = np.where(country_id_arr < 0, country_id_touched, country_id_arr)

priority_mask = to_array(biodiversity_da) >= 1

def get_mask(gid0=None, exposed=True):
    mask = np.ones(ref_shape, dtype=bool) if gid0 is None else (
        country_id_arr == countries_proj.index[countries_proj["GID_0"] == gid0][0]
    )
    return mask & priority_mask if exposed else mask

In [6]:
##### BUILD TABLE

def pct_by_percentile(var, pctl, mask):
    valid = ~np.isnan(var) & ~np.isnan(pctl) & (pctl >= 1) & (pctl <= 100) & mask
    values = var[valid]
    total = values.sum()
    sums = np.bincount(np.round(pctl[valid]).astype(int), weights=values, minlength=101)[1:101]
    return 100 * sums / total if total > 0 else sums

REGION_SPECS = [
    ("global",   "Global", None),
    ("national", "Global", None),
    ("national", "ETH",    "ETH"),
    ("national", "BOL",    "BOL"),
    ("national", "AUS",    "AUS"),
]

rows = []
for basis, region, gid0 in REGION_SPECS:
    pctl = pctl_arrays[basis]
    for area, exposed in [("overall", False), ("exposed", True)]:
        mask = get_mask(gid0, exposed)
        for variable, estimates in VARIABLES.items():
            for estimate, arr in estimates.items():
                pct = pct_by_percentile(arr, pctl, mask)
                rows.append(pd.DataFrame({
                    "pctl_basis": basis,
                    "region": region,
                    "variable": variable,
                    "area": area,
                    "estimate": estimate,
                    "percentile": np.arange(1, 101),
                    "pct_of_total": pct,
                }))

result_df = pd.concat(rows, ignore_index=True)

##### SAVE
result_df.to_csv(RESULTS_DIR / "exposure_by_deprivation_percentile.csv", index=False)